# Вращение векторов в 2D и 3D

В этом конспекте:

- Вращение вектора в 2D и матрица поворота.
- Вращение вектора в 3D вокруг осей $X$, $Y$, $Z$.
- Вращение вокруг произвольной оси: формула Родрига.
- Проверки свойств матриц вращения (ортогональность, $\det=1$, сохранение длины).
- Матрицы в ML как набор векторов и “вращение матрицы”
- Практические примеры на **NumPy** с комментариями в коде и выводами через `print`.

Обозначения:

- Вектор в 2D: $\mathbf{v} = (x, y)^T$
- Вектор в 3D: $\mathbf{v} = (x, y, z)^T$
- Угол поворота: $\theta$ (в радианах)


## 1) Вращение вектора в 2D

Пусть задан вектор

$$
\mathbf{v} =
\begin{pmatrix}
x \\
y
\end{pmatrix}
$$

Хотим повернуть его **против часовой стрелки** на угол $\theta$.

### Матрица поворота в 2D

$$
R(\theta) =
\begin{pmatrix}
\cos \theta & -\sin \theta \\
\sin \theta & \cos \theta
\end{pmatrix}
$$

### Формулы для новых координат

$$
\begin{pmatrix}
x' \\
y'
\end{pmatrix}
=
\begin{pmatrix}
\cos \theta & -\sin \theta \\
\sin \theta & \cos \theta
\end{pmatrix}
\begin{pmatrix}
x \\
y
\end{pmatrix}
$$

Отсюда:

$$
x' = x\cos\theta - y\sin\theta
$$

$$
y' = x\sin\theta + y\cos\theta
$$

### Поворот по часовой стрелке

Поворот по часовой стрелке на угол $\theta$ — это поворот на угол $-\theta$:

$$
R(-\theta) =
\begin{pmatrix}
\cos \theta & \sin \theta \\
-\sin \theta & \cos \theta
\end{pmatrix}
$$


In [1]:
import numpy as np

# --- 2D: функции для матрицы поворота и применения к вектору ---
def rot2d(theta: float) -> np.ndarray:
    """Матрица поворота в 2D на угол theta (радианы), против часовой стрелки."""
    c = np.cos(theta)  # cos(theta)
    s = np.sin(theta)  # sin(theta)
    return np.array([[c, -s],
                     [s,  c]], dtype=float)

def rotate_vector_2d(v: np.ndarray, theta: float) -> np.ndarray:
    """Поворачивает 2D-вектор v на угол theta (радианы), против часовой стрелки."""
    R = rot2d(theta)        # матрица поворота
    return R @ v            # (2x2) @ (2,) -> (2,)

# --- пример: поворот (1, 0) на 90 градусов ---
v = np.array([1.0, 0.0])            # исходный вектор
theta = np.deg2rad(90)              # 90° -> радианы

v_rot = rotate_vector_2d(v, theta)  # поворачиваем

print("Исходный вектор v:", v)
print("Угол (градусы):", 90)
print("Матрица R(θ):\n", rot2d(theta))
print("Повернутый вектор v':", v_rot)

# Вывод: ожидаем близко к (0, 1).


Исходный вектор v: [1. 0.]
Угол (градусы): 90
Матрица R(θ):
 [[ 6.123234e-17 -1.000000e+00]
 [ 1.000000e+00  6.123234e-17]]
Повернутый вектор v': [6.123234e-17 1.000000e+00]


In [2]:
# --- 2D: проверим, что длина сохраняется ---
def norm(v: np.ndarray) -> float:
    """Евклидова норма (длина) вектора."""
    return float(np.linalg.norm(v))

v2 = np.array([3.0, 4.0])           # длина = 5
theta2 = np.deg2rad(37)             # произвольный угол

v2_rot = rotate_vector_2d(v2, theta2)

print("v:", v2, "||v|| =", norm(v2))
print("v':", v2_rot, "||v'|| =", norm(v2_rot))
print("Разница норм:", abs(norm(v2_rot) - norm(v2)))

# Вывод: разница должна быть очень маленькой (численные погрешности float).


v: [3. 4.] ||v|| = 5.0
v': [-0.01135356  4.99998711] ||v'|| = 5.0
Разница норм: 0.0


In [3]:
# --- 2D: массовое вращение набора векторов (батч) ---
# Допустим, у нас N точек/векторов, каждая строка - вектор (x, y)
V = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [-2.0, 3.0],
], dtype=float)

theta = np.deg2rad(30)          # угол 30°
R = rot2d(theta)                # (2x2)

# Чтобы повернуть все строки, удобно умножить справа на R^T:
# V_rot[i] = R @ V[i]^T  эквивалентно  V_rot = V @ R^T
V_rot = V @ R.T

print("V (исходные векторы):\n", V)
print("R(θ):\n", R)
print("V_rot (повернутые):\n", V_rot)

# Вывод: каждый вектор из V повернулся на 30° против часовой стрелки.


V (исходные векторы):
 [[ 1.  0.]
 [ 0.  1.]
 [ 1.  1.]
 [-2.  3.]]
R(θ):
 [[ 0.8660254 -0.5      ]
 [ 0.5        0.8660254]]
V_rot (повернутые):
 [[ 0.8660254   0.5       ]
 [-0.5         0.8660254 ]
 [ 0.3660254   1.3660254 ]
 [-3.23205081  1.59807621]]


## 2) Вращение вектора в 3D

В 3D вращение задаётся:

- углом $\theta$
- осью вращения

Самый частый базовый случай — вращение вокруг координатных осей $X$, $Y$, $Z$.

Пусть

$$
\mathbf{v} =
\begin{pmatrix}
x \\
y \\
z
\end{pmatrix}
$$


### 2.1) Вращение вокруг оси $X$

Матрица:

$$
R_x(\theta)=
\begin{pmatrix}
1 & 0 & 0 \\
0 & \cos\theta & -\sin\theta \\
0 & \sin\theta & \cos\theta
\end{pmatrix}
$$

Формулы:

$$
x' = x
$$

$$
y' = y\cos\theta - z\sin\theta
$$

$$
z' = y\sin\theta + z\cos\theta
$$

Интуитивно: вращается плоскость $yz$, а координата $x$ не меняется.


### 2.2) Вращение вокруг оси $Y$

Матрица:

$$
R_y(\theta)=
\begin{pmatrix}
\cos\theta & 0 & \sin\theta \\
0 & 1 & 0 \\
-\sin\theta & 0 & \cos\theta
\end{pmatrix}
$$

Формулы:

$$
x' = x\cos\theta + z\sin\theta
$$

$$
y' = y
$$

$$
z' = -x\sin\theta + z\cos\theta
$$

Интуитивно: вращается плоскость $xz$, а координата $y$ не меняется.


### 2.3) Вращение вокруг оси $Z$

Матрица:

$$
R_z(\theta)=
\begin{pmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{pmatrix}
$$

Формулы:

$$
x' = x\cos\theta - y\sin\theta
$$

$$
y' = x\sin\theta + y\cos\theta
$$

$$
z' = z
$$

Это тот же поворот, что и в 2D, но в плоскости $xy$.


In [4]:
import numpy as np

# --- 3D: матрицы поворота вокруг координатных осей ---
def rotx(theta: float) -> np.ndarray:
    """Вращение вокруг оси X на угол theta (радианы), против часовой стрелки по правилу правой руки."""
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[1.0, 0.0, 0.0],
                     [0.0,   c,  -s],
                     [0.0,   s,   c]], dtype=float)

def roty(theta: float) -> np.ndarray:
    """Вращение вокруг оси Y на угол theta (радианы), против часовой стрелки по правилу правой руки."""
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[  c, 0.0,  s],
                     [0.0, 1.0, 0.0],
                     [ -s, 0.0,  c]], dtype=float)

def rotz(theta: float) -> np.ndarray:
    """Вращение вокруг оси Z на угол theta (радианы), против часовой стрелки по правилу правой руки."""
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[  c,  -s, 0.0],
                     [  s,   c, 0.0],
                     [0.0, 0.0, 1.0]], dtype=float)

# --- пример: повернем вектор вокруг Z на 90° ---
v = np.array([1.0, 0.0, 5.0])          # вектор в 3D
theta = np.deg2rad(90)

v_rot = rotz(theta) @ v

print("v:", v)
print("Rz(90°):\n", rotz(theta))
print("v' = Rz @ v:", v_rot)

# Вывод: x,y должны повернуться как в 2D, а z остаться равным 5.


v: [1. 0. 5.]
Rz(90°):
 [[ 6.123234e-17 -1.000000e+00  0.000000e+00]
 [ 1.000000e+00  6.123234e-17  0.000000e+00]
 [ 0.000000e+00  0.000000e+00  1.000000e+00]]
v' = Rz @ v: [6.123234e-17 1.000000e+00 5.000000e+00]


In [5]:
# --- пример: сравним повороты вокруг X, Y, Z ---
v = np.array([1.0, 2.0, 3.0], dtype=float)
theta = np.deg2rad(30)

vx = rotx(theta) @ v
vy = roty(theta) @ v
vz = rotz(theta) @ v

print("Исходный v:", v)
print("Rx(30°) @ v =", vx)
print("Ry(30°) @ v =", vy)
print("Rz(30°) @ v =", vz)

# Вывод: разные оси => разные результаты (это ключевая разница с 2D).


Исходный v: [1. 2. 3.]
Rx(30°) @ v = [1.         0.23205081 3.59807621]
Ry(30°) @ v = [2.3660254  2.         2.09807621]
Rz(30°) @ v = [-0.1339746   2.23205081  3.        ]


## 3) Вращение вокруг произвольной оси: формула Родрига

Пусть ось вращения задана **единичным** вектором

$$
\mathbf{u} =
\begin{pmatrix}
u_x \\
u_y \\
u_z
\end{pmatrix}, \quad \|\mathbf{u}\| = 1
$$

Тогда матрица вращения на угол $\theta$ задаётся формулой Родрига:

$$
R = I\cos\theta + (1-\cos\theta)\,\mathbf{u}\mathbf{u}^T + [\mathbf{u}]_\times \sin\theta
$$

Где:

- $I$ — единичная матрица $3\times 3$
- $\mathbf{u}\mathbf{u}^T$ — внешнее произведение (матрица ранга 1)
- $[\mathbf{u}]_\times$ — матрица, реализующая векторное произведение с $\mathbf{u}$:

$$
[\mathbf{u}]_\times =
\begin{pmatrix}
0 & -u_z & u_y \\
u_z & 0 & -u_x \\
-u_y & u_x & 0
\end{pmatrix}
$$

### Явная запись матрицы

$$
R =
\begin{pmatrix}
\cos\theta + u_x^2(1-\cos\theta) &
u_xu_y(1-\cos\theta) - u_z\sin\theta &
u_xu_z(1-\cos\theta) + u_y\sin\theta \\
\\
u_yu_x(1-\cos\theta) + u_z\sin\theta &
\cos\theta + u_y^2(1-\cos\theta) &
u_yu_z(1-\cos\theta) - u_x\sin\theta \\
\\
u_zu_x(1-\cos\theta) - u_y\sin\theta &
u_zu_y(1-\cos\theta) + u_x\sin\theta &
\cos\theta + u_z^2(1-\cos\theta)
\end{pmatrix}
$$


In [6]:
import numpy as np

def normalize(u: np.ndarray) -> np.ndarray:
    """Нормирует вектор u до единичной длины."""
    u = np.asarray(u, dtype=float)
    n = np.linalg.norm(u)
    if n == 0:
        raise ValueError("Нельзя нормировать нулевой вектор (ось вращения должна быть ненулевой).")
    return u / n

def skew(u: np.ndarray) -> np.ndarray:
    """Кососимметричная матрица [u]_x для векторного произведения: [u]_x v = u × v."""
    ux, uy, uz = u
    return np.array([[0.0, -uz,  uy],
                     [ uz, 0.0, -ux],
                     [-uy,  ux, 0.0]], dtype=float)

def rodrigues(axis: np.ndarray, theta: float) -> np.ndarray:
    """Матрица вращения вокруг произвольной оси axis на угол theta (радианы) по формуле Родрига."""
    u = normalize(axis)         # ось должна быть единичной
    I = np.eye(3, dtype=float)  # единичная матрица
    c = np.cos(theta)
    s = np.sin(theta)

    uuT = np.outer(u, u)        # u u^T
    K = skew(u)                 # [u]_x

    R = I * c + (1.0 - c) * uuT + s * K
    return R

# --- пример: вращение вокруг оси (1, 1, 1) на 60° ---
axis = np.array([1.0, 1.0, 1.0])
theta = np.deg2rad(60)

R = rodrigues(axis, theta)

v = np.array([2.0, -1.0, 0.5])
v_rot = R @ v

print("Ось (до нормировки):", axis)
print("Ось (единичная):", normalize(axis))
print("Угол (градусы):", 60)
print("Матрица Родрига R:\n", R)
print("v:", v)
print("R @ v:", v_rot)


Ось (до нормировки): [1. 1. 1.]
Ось (единичная): [0.57735027 0.57735027 0.57735027]
Угол (градусы): 60
Матрица Родрига R:
 [[ 0.66666667 -0.33333333  0.66666667]
 [ 0.66666667  0.66666667 -0.33333333]
 [-0.33333333  0.66666667  0.66666667]]
v: [ 2.  -1.   0.5]
R @ v: [ 2.   0.5 -1. ]


## 4) Свойства матриц вращения (важные проверки)

Любая матрица вращения $R$ в 2D или 3D обладает свойствами:

1) **Ортогональность**:

$$
R^T R = I
$$

2) **Определитель равен 1**:

$$
\det(R)=1
$$

3) **Сохранение длины** (евклидовой нормы):

$$
\|R\mathbf{v}\| = \|\mathbf{v}\|
$$

Геометрически это означает: вращение не растягивает и не сжимает пространство, а только меняет направление.


In [7]:
# --- проверим свойства для 3D-матрицы Родрига ---
axis = np.array([0.2, 0.9, -0.4])
theta = np.deg2rad(123)

R = rodrigues(axis, theta)

# Ортогональность: R^T R должно быть близко к I
I_approx = R.T @ R
I = np.eye(3)

# Определитель: должен быть близко к 1
detR = np.linalg.det(R)

# Сохранение длины для нескольких случайных векторов
rng = np.random.default_rng(42)
V = rng.normal(size=(5, 3))          # 5 случайных векторов
V_rot = (R @ V.T).T                  # поворачиваем все (через транспонирование)

norms_before = np.linalg.norm(V, axis=1)
norms_after  = np.linalg.norm(V_rot, axis=1)

print("R^T R:\n", I_approx)
print("Насколько R^T R близко к I (макс. отклонение):", np.max(np.abs(I_approx - I)))
print("det(R) =", detR)
print("Нормы до:", norms_before)
print("Нормы после:", norms_after)
print("Максимальная разница норм:", np.max(np.abs(norms_after - norms_before)))

# Вывод: отклонения должны быть малы (из-за округления float).


R^T R:
 [[ 1.00000000e+00 -2.89162843e-16  1.04685532e-16]
 [-2.89162843e-16  1.00000000e+00  4.49759392e-16]
 [ 1.04685532e-16  4.49759392e-16  1.00000000e+00]]
Насколько R^T R близко к I (макс. отклонение): 1.3322676295501878e-15
det(R) = 0.9999999999999989
Нормы до: [1.31817921 2.5272261  0.34151841 1.45120124 1.22212838]
Нормы после: [1.31817921 2.5272261  0.34151841 1.45120124 1.22212838]
Максимальная разница норм: 8.881784197001252e-16


## 5) Комбинация вращений в 3D (важное отличие от 2D)

В 2D повороты всегда “живут” вокруг одной оси (перпендикулярной плоскости), поэтому композиция поворотов проста:

$$
R(\alpha)R(\beta) = R(\alpha+\beta)
$$

В 3D повороты вокруг **разных** осей обычно **не коммутируют**:

$$
R_x(\alpha)R_y(\beta) \ne R_y(\beta)R_x(\alpha)
$$

То есть порядок умножения матриц важен.


In [8]:
# --- покажем, что в 3D порядок вращений важен ---
v = np.array([1.0, 2.0, 3.0])
a = np.deg2rad(45)   # угол 45°
b = np.deg2rad(30)   # угол 30°

Rxy = rotx(a) @ roty(b)   # сначала вокруг Y, потом вокруг X (умножение справа-на-лево)
Ryx = roty(b) @ rotx(a)   # сначала вокруг X, потом вокруг Y

v1 = Rxy @ v
v2 = Ryx @ v

print("v:", v)
print("Rxy @ v:", v1)
print("Ryx @ v:", v2)
print("Разница:", v1 - v2)

# Вывод: результаты обычно различаются => повороты вокруг разных осей не коммутируют.


v: [1. 2. 3.]
Rxy @ v: [ 2.3660254  -0.06935035  2.89777748]
Ryx @ v: [ 2.63379236 -0.70710678  2.56186218]
Разница: [-0.26776695  0.63775643  0.3359153 ]


## 6) Матрицы в ML как набор векторов и “вращение матрицы”

В машинном обучении почти всегда встречается матрица данных:

- $X \in \mathbb{R}^{n\times d}$ — **матрица признаков** (feature matrix),
- $n$ — число объектов (samples),
- $d$ — число признаков (features).

Обычно принимают соглашение:

- **строка** $x_i$ — один объект (одна выборка),
- **столбец** — один признак.

То есть $X$ можно воспринимать как **набор векторов-строк**:

$$
X =
\begin{pmatrix}
- \\ x_1 \\ -
\\
- \\ x_2 \\ -
\\
\vdots
\\
- \\ x_n \\ -
\end{pmatrix}
$$

### 1) “Вращение матрицы” = одинаковое вращение множества векторов

Если у нас есть матрица поворота (или, в общем случае, ортогональная матрица) $R \in \mathbb{R}^{d\times d}$, то:

- при соглашении **векторы-столбцы** (классическая линейная алгебра):
  $$
  x' = R x
  $$
- при соглашении **векторы-строки** (часто в ML/NumPy датасеты):
  $$
  x' = x R^T
  $$

Почему появляется транспонирование? Потому что запись $xR$ соответствует умножению **справа** для вектора-строки, а стандартная формула $Rx$ — это умножение **слева** для вектора-столбца.

Для всей матрицы данных (как набора строк-векторов) получаем:

$$
X' = X R^T
$$

Это означает: **каждая строка $x_i$ поворачивается одинаково**.

### 2) Левое и правое умножение — разные смыслы

- $X' = X R^T$ (векторы-строки) — мы “вращаем сами точки/объекты” в пространстве признаков.
- $X' = R X$ (векторы-столбцы, если столбцы — точки) — тоже вращение, но для другой упаковки данных.

Важно: “поворот матрицы” в таком смысле — это НЕ то же самое, что `np.rot90`, который просто переставляет элементы в таблице (это уже про *картинки/массивы*, а не про линейное преобразование пространства).

---

Ниже — практические примеры на NumPy, которые показывают:
1) как вращать сразу весь набор точек одним умножением матриц,
2) как проверить, что это эквивалентно вращению каждой точки по отдельности.


In [9]:
# --- Пример 1: 2D. Вращаем сразу N точек (каждая точка = строка) ---
np.random.seed(42)

N = 5
X = np.random.randn(N, 2)  # N точек на плоскости (N,2)

theta = np.deg2rad(30)  # 30 градусов
R2 = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

# Соглашение ML/NumPy: каждая точка - строка -> X' = X @ R^T
X_rot = X @ R2.T

print("Исходные точки X (строки):")
print(X)
print("\nМатрица поворота R2:")
print(R2)
print("\nПовернутые точки X_rot = X @ R2.T:")
print(X_rot)

# Проверим, что это то же самое, что повернуть каждую точку отдельно
X_rot_loop = np.vstack([R2 @ x for x in X])  # здесь x - вектор-столбец "по смыслу", поэтому R2 @ x

print("\nПовернутые точки через цикл (R2 @ x):")
print(X_rot_loop)

print("\nРазница (должна быть ~0):")
print(np.max(np.abs(X_rot - X_rot_loop)))

# --- Проверка инвариантности длины (вращение сохраняет нормы) ---
norms_before = np.linalg.norm(X, axis=1)
norms_after = np.linalg.norm(X_rot, axis=1)

print("\nНормы до поворота:", norms_before)
print("Нормы после поворота:", norms_after)
print("Макс. отклонение норм:", np.max(np.abs(norms_before - norms_after)))


Исходные точки X (строки):
[[ 0.49671415 -0.1382643 ]
 [ 0.64768854  1.52302986]
 [-0.23415337 -0.23413696]
 [ 1.57921282  0.76743473]
 [-0.46947439  0.54256004]]

Матрица поворота R2:
[[ 0.8660254 -0.5      ]
 [ 0.5        0.8660254]]

Повернутые точки X_rot = X @ R2.T:
[[ 0.49929923  0.12861668]
 [-0.2006002   1.64282682]
 [-0.08571429 -0.31984524]
 [ 0.98392105  1.45422438]
 [-0.67785677  0.23513359]]

Повернутые точки через цикл (R2 @ x):
[[ 0.49929923  0.12861668]
 [-0.2006002   1.64282682]
 [-0.08571429 -0.31984524]
 [ 0.98392105  1.45422438]
 [-0.67785677  0.23513359]]

Разница (должна быть ~0):
1.1102230246251565e-16

Нормы до поворота: [0.51559865 1.65502882 0.33113127 1.75581012 0.71748003]
Нормы после поворота: [0.51559865 1.65502882 0.33113127 1.75581012 0.71748003]
Макс. отклонение норм: 0.0


In [10]:
# --- Пример 2: 3D. Вращаем N точек вокруг оси Z ---
np.random.seed(0)

N = 5
X3 = np.random.randn(N, 3)  # N точек в 3D (N,3)

phi = np.deg2rad(45)  # 45 градусов
Rz = rotz(phi)        # используем функцию rotz из конспекта выше

# Каждая точка - строка -> X3' = X3 @ Rz.T
X3_rot = X3 @ Rz.T

print("Исходные точки X3 (строки):")
print(X3)
print("\nМатрица поворота вокруг Z (Rz):")
print(Rz)
print("\nПовернутые точки X3_rot = X3 @ Rz.T:")
print(X3_rot)

# Эквивалентность с покомпонентным вращением каждой точки (Rz @ x)
X3_rot_loop = np.vstack([Rz @ x for x in X3])

print("\nПовернутые точки через цикл (Rz @ x):")
print(X3_rot_loop)

print("\nМакс. разница (должна быть ~0):", np.max(np.abs(X3_rot - X3_rot_loop)))

# --- Длины тоже сохраняются ---
norms3_before = np.linalg.norm(X3, axis=1)
norms3_after = np.linalg.norm(X3_rot, axis=1)

print("\nНормы до поворота:", norms3_before)
print("Нормы после поворота:", norms3_after)
print("Макс. отклонение норм:", np.max(np.abs(norms3_before - norms3_after)))


Исходные точки X3 (строки):
[[ 1.76405235  0.40015721  0.97873798]
 [ 2.2408932   1.86755799 -0.97727788]
 [ 0.95008842 -0.15135721 -0.10321885]
 [ 0.4105985   0.14404357  1.45427351]
 [ 0.76103773  0.12167502  0.44386323]]

Матрица поворота вокруг Z (Rz):
[[ 0.70710678 -0.70710678  0.        ]
 [ 0.70710678  0.70710678  0.        ]
 [ 0.          0.          1.        ]]

Повернутые точки X3_rot = X3 @ Rz.T:
[[ 0.9644195   1.53032725  0.97873798]
 [ 0.26398786  2.9051137  -0.97727788]
 [ 0.77883967  0.56478825 -0.10321885]
 [ 0.1884828   0.39219117  1.45427351]
 [ 0.45209771  0.62417217  0.44386323]]

Повернутые точки через цикл (Rz @ x):
[[ 0.9644195   1.53032725  0.97873798]
 [ 0.26398786  2.9051137  -0.97727788]
 [ 0.77883967  0.56478825 -0.10321885]
 [ 0.1884828   0.39219117  1.45427351]
 [ 0.45209771  0.62417217  0.44386323]]

Макс. разница (должна быть ~0): 1.1102230246251565e-16

Нормы до поворота: [2.05668046 3.07643417 0.96759038 1.51797599 0.88938057]
Нормы после поворота: [

### 3) Где это встречается в ML на практике

- **PCA**: по сути выбирает новый ортонормированный базис (поворот/отражение пространства признаков) и затем проецирует данные.
- **Whitening** (например, ZCA / PCA whitening): сначала поворот в базис главных компонент, затем масштабирование (это уже не чистый поворот).
- **Embeddings / линейный слой**: общий случай $XW$ — это не только поворот, но и растяжение/сжатие/сдвиг (если есть bias).
- **Data augmentation** для координат/точек (робототехника, компьютерное зрение, 3D): поворот облака точек делается ровно так же: `points @ R.T`.

Главное, что стоит помнить про NumPy формы:

- если точки хранятся как строки `(N, d)`, то “применить матрицу преобразования” чаще всего означает `X @ R.T`;
- если вы храните точки как столбцы `(d, N)`, то это будет `R @ X`.

И это одна и та же геометрия — просто разные соглашения записи.
